1. Download PDFs and preprocess

In [ ]:
# Convert pdf to txt 

import PyPDF2

def pdf_to_text(pdf_path, output_txt):
    #open pdf in binary
    with open(pdf_path, 'rb') as pdf_file:
        #create object that allows page reading
        pdf_reader = PyPDF2.PdfReader(pdf_file)

        text = ''

        for page_num in range(len(pdf_reader.pages)): #go throught every page
            page = pdf_reader.pages[page_num] #get page with no [page_num]
            text += page.extract_text() #get text from page

    #save to txt file
    with open(output_txt, 'w', encoding='utf-8') as txt_file:
        txt_file.write(text)

pdf_to_text('5 A Dance With Dragons.pdf', 'Book_5.txt') #initialise function

In [ ]:
#remove cringe formatting
with open("Book_1.txt", "r") as file, open("Clean_Book_1.txt", 'w', encoding='utf-8') as outfile:
    for line in file:
        if "Table of Contents" not in line: #delete formating in book1
            outfile.write(line)

In [ ]:
#connect books
import os
#books to combine
books_to_combine = ["Clean_Book_1.txt", "Book_2.txt", "Book_3.txt", "Book_4.txt","Book_5.txt"] 
output = "All_books.txt"

with open(output, "w", encoding="utf-8") as outfile:
    for file_name in books_to_combine:
        #open each gile
        with open(file_name, "r", encoding="utf-8") as infile:
            #write contents
            outfile.write(infile.read())

2. Tokenize

In [ ]:
#tokenize
import nltk 
from nltk.tokenize import sent_tokenize, word_tokenize
import statistics as st
from nltk import FreqDist
import re

with open("All_books.txt", "r") as file:
    books = file.read().replace('\n',' ').lower()

#treat "..." as one word
books = re.sub(r"\. \. \.", " TRZYKROPEKK", books)
books =re.sub(r" ’", "'", books) #fix cringe formatting
books =re.sub(r"’", "'", books) #fix cringe formatting

from nltk.tokenize import RegexpTokenizer
# define tokenizer: words + shortenings (e.g don't) as token
tokenizer = RegexpTokenizer("[A-Za-z]+'?[A-Za-z]*") 

#tokenize
sentences = sent_tokenize(books)
words_in_sents = [tokenizer.tokenize(sent) for sent in sentences]

#restore placeholder to "..."
words_in_sents = [[word.replace("TRZYKROPEKK", "...") for word in sent] for sent in words_in_sents]

3. Tag words

In [33]:
#contractions can be incorectly tagged so we perform additional tagging and cleaning
#below lists are genereated by chat gbt

#ambiguous tags dont have an equivalent in pos_tag so we delete those words 
ambiguous_tags = [
    "he's", "she's", "it's", "that's", "this's", "somebody's", "anybody's", "everybody's", 
    "i'll", "you'll", "he'll", "she'll", "it'll", "we'll", "they'll", 
    "i'd", "you'd", "he'd", "she'd", "it'd", "we'd", "they'd", 
    "i've", "you've", "we've", "they've", "we're", "you're", "they're", "i'd've", "you'd've", 
    "he'd've", "she'd've", "it'd've", "we'd've", "they'd've", "should've", "would've", "could've", 
    "must've", "might've", "may've", "ought've"]

#correct_tags have proper tags but nltk stuggles with idetyfying them
contractions = [
    "aren't", "can't", "couldn't", "daren't", "didn't", "doesn't", "don't",
    "hadn't", "hasn't", "haven't", "isn't", "mustn't", "needn't", "shan't",
    "shouldn't", "wasn't", "weren't", "won't", "wouldn't", "ain't",
    "mightn't", "mayn't", "usedn't", "oughtn't",  # Negations
    "who's", "what's", "where's", "when's", "why's", "how's",  # Wh-words
    "there's", "here's", "let's",  # Common structures
]
contractions_tagged = [
    ("aren't", "VBP"), ("can't", "MD"),  ("couldn't", "MD"),  ("daren't", "MD"),  ("didn't", "VBD"),  
    ("doesn't", "VBZ"), ("don't", "VBP"),  ("hadn't", "VBD"),  ("hasn't", "VBZ"),  ("haven't", "VBP"),  
    ("isn't", "VBZ"), ("mustn't", "MD"),  ("needn't", "MD"),  ("shan't", "MD"),  ("shouldn't", "MD"),  
    ("wasn't", "VBD"), ("weren't", "VBD"),  ("won't", "MD"), ("wouldn't", "MD"), ("ain't", "VBZ"),
    ("mightn't", "MD"), ("mayn't", "MD"),  ("usedn't", "VBD"),  ("oughtn't", "MD"), ("who's", "WP$"), 
    ("what's", "WP$"), ("where's", "WRB"), ("when's", "WRB"),  ("why's", "WRB"),  ("how's", "WRB"),  
    ("there's", "EX"), ("here's", "RB"),  ("let's", "VB") 
]
#from my research nltk works resonably well with possesive nouns

In [34]:
#delete ambiguous tags form words
words_in_sents=[[w for w in sent if w not in ambiguous_tags] for sent in words_in_sents]

In [35]:
#get tags
from nltk import pos_tag
tags = [pos_tag(sent) for sent in words_in_sents]
print(tags[:10])

[[('prologue', 'NN'), ('we', 'PRP'), ('should', 'MD'), ('start', 'VB'), ('back', 'RB'), ('gared', 'JJ'), ('urged', 'VBD'), ('as', 'IN'), ('the', 'DT'), ('woods', 'NNS'), ('began', 'VBD'), ('to', 'TO'), ('grow', 'VB'), ('dark', 'NN'), ('around', 'IN'), ('them', 'PRP')], [('the', 'DT'), ('wildlings', 'NNS'), ('are', 'VBP'), ('dead', 'JJ'), ('do', 'VBP'), ('the', 'DT'), ('dead', 'JJ'), ('frighten', 'NN'), ('you', 'PRP'), ('ser', 'VBP'), ('waymar', 'JJ'), ('royce', 'NN'), ('asked', 'VBD'), ('with', 'IN'), ('just', 'RB'), ('the', 'DT'), ('hint', 'NN'), ('of', 'IN'), ('a', 'DT'), ('smile', 'NN'), ('gared', 'VBN'), ('did', 'VBD'), ('not', 'RB'), ('rise', 'VB'), ('to', 'TO'), ('the', 'DT'), ('bait', 'NN')], [('he', 'PRP'), ('was', 'VBD'), ('an', 'DT'), ('old', 'JJ'), ('man', 'NN'), ('past', 'IN'), ('fifty', 'NN'), ('and', 'CC'), ('he', 'PRP'), ('had', 'VBD'), ('seen', 'VBN'), ('the', 'DT'), ('lordlings', 'NNS'), ('come', 'VBP'), ('and', 'CC'), ('go', 'VBP')], [('dead', 'JJ'), ('is', 'VBZ'), ('

In [36]:
#function to fix the contraction tags
def correct_tags(tags, contractions_tagged):
    #convert contractions_tagged to a dictionary
    contraction_dict = dict(contractions_tagged)
    
    #correct the pos tags where needed
    corrected_tags = [
        [
            (word, contraction_dict[word]) if word in contraction_dict else (word, pos)
            for word, pos in sentence
        ]
        for sentence in tags
    ]
    
    return corrected_tags

corrected_tags = correct_tags(tags, contractions_tagged)
print(corrected_tags[:50])

[[('prologue', 'NN'), ('we', 'PRP'), ('should', 'MD'), ('start', 'VB'), ('back', 'RB'), ('gared', 'JJ'), ('urged', 'VBD'), ('as', 'IN'), ('the', 'DT'), ('woods', 'NNS'), ('began', 'VBD'), ('to', 'TO'), ('grow', 'VB'), ('dark', 'NN'), ('around', 'IN'), ('them', 'PRP')], [('the', 'DT'), ('wildlings', 'NNS'), ('are', 'VBP'), ('dead', 'JJ'), ('do', 'VBP'), ('the', 'DT'), ('dead', 'JJ'), ('frighten', 'NN'), ('you', 'PRP'), ('ser', 'VBP'), ('waymar', 'JJ'), ('royce', 'NN'), ('asked', 'VBD'), ('with', 'IN'), ('just', 'RB'), ('the', 'DT'), ('hint', 'NN'), ('of', 'IN'), ('a', 'DT'), ('smile', 'NN'), ('gared', 'VBN'), ('did', 'VBD'), ('not', 'RB'), ('rise', 'VB'), ('to', 'TO'), ('the', 'DT'), ('bait', 'NN')], [('he', 'PRP'), ('was', 'VBD'), ('an', 'DT'), ('old', 'JJ'), ('man', 'NN'), ('past', 'IN'), ('fifty', 'NN'), ('and', 'CC'), ('he', 'PRP'), ('had', 'VBD'), ('seen', 'VBN'), ('the', 'DT'), ('lordlings', 'NNS'), ('come', 'VBP'), ('and', 'CC'), ('go', 'VBP')], [('dead', 'JJ'), ('is', 'VBZ'), ('

In [37]:
import pickle
with open("corrected_tags.pkl", 'wb') as file:
    pickle.dump(corrected_tags, file)

4. Check lengths probabilities for sentences

In [38]:
import nltk 
from nltk import FreqDist
# sent length with prob
avg_sent_length = FreqDist([len(sent) for sent in words_in_sents]).most_common(12)
sent_sum = sum(list(zip(*avg_sent_length))[1]) # sent sum (i want probs to add add up to 1)
avg_sent_length = [(sent, freq/sent_sum) for sent, freq in avg_sent_length] #get freq of each sent length
print(avg_sent_length)

[(6, 0.10313022503631829), (7, 0.10036181235095798), (5, 0.09376970095660993), (8, 0.09313927034509223), (9, 0.08791766028013047), (4, 0.08255900008223008), (10, 0.08228489981635283), (11, 0.08080475838061563), (12, 0.07480196255790368), (13, 0.07114272400844229), (14, 0.0681824411369679), (15, 0.061905545048378696)]


In [17]:
print(list(zip(*avg_sent_length))[0])

(6, 7, 5, 8, 9, 4, 10, 11, 12, 13, 14, 15)


In [39]:
import pickle
with open("avg_sent_length.pkl", 'wb') as file:
    pickle.dump(avg_sent_length, file)

5. N-grams

In [40]:
#n grams

from nltk import ngrams
words2 = [w for sent in words_in_sents for w in sent]
n_grams = list(ngrams(words2,3))

ngrams_freq = FreqDist(n_grams).most_common(1000000)
ngrams_freq_sum = sum(freq for ngram, freq in ngrams_freq) #ngram greq sum (i want probs to add add up to 1)
ngrams_prob = {gram: freq / ngrams_freq_sum for gram, freq in ngrams_freq} #create a dict of ngrams and freqs

In [41]:
with open("ngrams_prob.pkl", 'wb') as file:
    pickle.dump(ngrams_prob, file)

7.1. HMM model
-   bulit the model

In [ ]:
#delete empty sentences
corrected_tags = [sent for sent in corrected_tags if sent]

train_set =  corrected_tags[:100000]
test_set =   corrected_tags[100000:]

from nltk.tag import hmm

HMM_model = hmm.HiddenMarkovModelTrainer() 
HMM_tagger = HMM_model.train_supervised(train_set)

In [80]:
accuracy = HMM_tagger.evaluate(test_set)
print(f"Model Accuracy: {accuracy}")

C:\Users\kajaw\AppData\Local\Temp\ipykernel_4560\3883013956.py:1: DeprecationWarning: 
  Function evaluate() has been deprecated.  Use accuracy(gold)
  instead.
  accuracy = HMM_tagger.evaluate(test_set)
c:\Users\kajaw\miniconda3\envs\Pbioinf1.2\Lib\site-packages\nltk\tag\hmm.py:333: RuntimeWarning: overflow encountered in cast
  X[i, j] = self._transitions[si].logprob(self._states[j])
c:\Users\kajaw\miniconda3\envs\Pbioinf1.2\Lib\site-packages\nltk\tag\hmm.py:335: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])
c:\Users\kajaw\miniconda3\envs\Pbioinf1.2\Lib\site-packages\nltk\tag\hmm.py:331: RuntimeWarning: overflow encountered in cast
  P[i] = self._priors.logprob(si)
c:\Users\kajaw\miniconda3\envs\Pbioinf1.2\Lib\site-packages\nltk\tag\hmm.py:363: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])


Model Accuracy: 0.8457857413522438


In [81]:
import dill

#you cant pickle hmm model
with open("HMM_tagger.dill", "wb") as file:
    dill.dump(HMM_tagger, file)

-    genrate senteces

In [82]:
#generate a sentences
def generate_sentences(avg_sent_length, HMM_tagger, sents_num):
    import random

    #lengths of sentences
    lengths, probabilities = zip(*avg_sent_length)

    #list of tags that shouldnt appear at th eend of the sentence
    forbidden_end_tags = {"IN", "CC", "DT", "PRP$", "RP"}

    generated_sentences =[]
    while len(generated_sentences) < sents_num:
        #generate a sequence of words
        num_words = random.choices(lengths, weights=probabilities, k=1)[0]  #random choices generates a list even if k=1, hence [0]
        generate_sent = HMM_tagger.random_sample(random.Random(), num_words)
    
        #check if the last tag is not a forbidden tag
        if generate_sent[-1][1] in forbidden_end_tags:
            continue  
        
        #get only words and connect them
        generate_sent = " ".join(word for word, tag in generate_sent)
        generated_sentences.append(generate_sent)
        
    return generated_sentences
#the generated sentences are quite poor gramatically, 
#we generate 10 000 of them and then apply filtering techniques to find the best ones

In [83]:
#we generate 10 000 sentences and then apply filtering techniques to find the best ones

gen_sents = generate_sentences(avg_sent_length, HMM_tagger, 10000)

In [84]:
import pickle
with open("gen_sents.pkl", 'wb') as file:
    pickle.dump(gen_sents, file)

- apply ngram filtering

In [85]:
#function rating the sentence by ngrams probs
def ngram_score(sentence, ngrams_prob_dict, ngram_size):
    from nltk import ngrams

    words = sentence.split() #split sentences 
    n_grams = list(ngrams(words,ngram_size))  #generate ngrams for sentence

    if not n_grams: #if no ngrams generated (sentence shorter than ngram_size)
        return 0  
    
    score = sum(ngrams_prob_dict.get(ng, 0) for ng in n_grams)  # sum probabilities
    return score / len(n_grams)  #normalize by sentence length

-   rate the senteces

In [86]:
def rate_by_ngrams(gen_sents, ngrams_prob, ngram_size, number_of_sentences):
    gen_sents_probs = []
    for sent in gen_sents:
        prob = ngram_score(sent, ngrams_prob, 3)
        if prob > 0.000001:
            gen_sents_probs.append((sent, prob))

    gen_sents_probs = sorted(gen_sents_probs, key=lambda x: x[1], reverse=True)

    return gen_sents_probs[:number_of_sentences]

rate_by_ngrams(gen_sents, ngrams_prob, 3, 10)

[('how it was not', 7.229631499110099e-05),
 ('it was a waist put him', 4.8635702812195205e-05),
 ('not twisted it was the floor', 4.3049169381064676e-05),
 ('the of it was the orange grew', 3.785697948624924e-05),
 ('the tyrion after he had been', 3.5655228075156624e-05),
 ('theon is between the lord of the seastone already she',
  3.2861961359591356e-05),
 ('and he was treason', 3.187610251880361e-05),
 ('your skinny he could not a proof', 3.1153139368892605e-05),
 ("he had a woman's worked", 2.8480366511645837e-05),
 ('wolves of you sang out of the man', 2.8261286769248568e-05)]

In [87]:
import dill
with open("rate_by_ngrams.dill", "wb") as file:
    dill.dump(rate_by_ngrams, file)

7.2 HMM with trigrams

In [ ]:
#unigram
[[("word1", "POS1"), ("word2", "POS2"), ("word3", "POS3")],  #sent1
    [("word4", "POS4"), ("word5", "POS5")],                  #sent2
    ...]

#trigram (word and 2 two previous words)
[[(("word1", "START", "START"), "POS1"), (("word2", "word1", "START"), "POS2"), (("word3", "word2", "word1"), "POS3")],  
    [(("word4", "START", "START"), "POS4"), (("word5", "word4", "START"), "POS5")],
    ...
]

In [10]:
def trigram(sentences):
    tri_sentences = []
    
    for sent in sentences:
        tri_sent = []
        art_sent = [("START", "START"), ("START", "START")] + sent  #add artificial start tokens
        
        for i in range(2, len(art_sent)):
            tri_word = (art_sent[i][0], art_sent[i-1][0], art_sent[i-2][0])
            tri_sent.append((tri_word, art_sent[i][1]))  # (trigram, tag)
        
        tri_sentences.append(tri_sent)
    
    return tri_sentences

-   build model

In [12]:
#delete empty sentences
corrected_tags = [sent for sent in corrected_tags if sent]

train_set =  corrected_tags[:100000]
test_set =   corrected_tags[100000:]

trigram_train_set = trigram(train_set)
trigram_test_set = trigram(test_set)

from nltk.tag import hmm

HMM_model = hmm.HiddenMarkovModelTrainer() 
HMM_tri_tagger = HMM_model.train_supervised(trigram_train_set)

In [1]:
import dill
with open("HMM_tri_tagger.pkl", 'wb') as file:
    HMM_tri_tagger =dill.load(file)
    #dill.dump(HMM_tri_tagger, file)

UnsupportedOperation: read

In [8]:
accuracy = HMM_tagger.evaluate(trigram_test_set)
print(f"Model Accuracy: {accuracy}")

C:\Users\kajaw\AppData\Local\Temp\ipykernel_2644\348979702.py:1: DeprecationWarning: 
  Function evaluate() has been deprecated.  Use accuracy(gold)
  instead.
  accuracy = HMM_tagger.evaluate(trigram_test_set)
c:\Users\kajaw\miniconda3\envs\Pbioinf1.2\Lib\site-packages\nltk\tag\hmm.py:333: RuntimeWarning: overflow encountered in cast
  X[i, j] = self._transitions[si].logprob(self._states[j])
c:\Users\kajaw\miniconda3\envs\Pbioinf1.2\Lib\site-packages\nltk\tag\hmm.py:335: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])
c:\Users\kajaw\miniconda3\envs\Pbioinf1.2\Lib\site-packages\nltk\tag\hmm.py:331: RuntimeWarning: overflow encountered in cast
  P[i] = self._priors.logprob(si)
c:\Users\kajaw\miniconda3\envs\Pbioinf1.2\Lib\site-packages\nltk\tag\hmm.py:363: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])


KeyboardInterrupt: 

In [13]:
import dill

#you cant pickle hmm model
with open("HMM_tri_tagger.dill", "wb") as file:
    dill.dump(HMM_tri_tagger, file)

-   basic sentence genration

In [35]:
#generate a sentences
def gen_basic_tri_sents(avg_sent_length, HMM_tri_tagger, sents_num):
    import random

    #lengths of sentences
    lengths, probabilities = zip(*avg_sent_length)

    #list of tags that shouldnt appear at th eend of the sentence
    forbidden_end_tags = {"IN", "CC", "DT", "PRP$", "RP"}

    generated_sentences =[]
    while len(generated_sentences) < sents_num:
        #generate a sequence of words
        num_words = random.choices(lengths, weights=probabilities, k=1)[0]  #random choices generates a list even if k=1, hence [0]
        generate_sent = HMM_tagger.random_sample(random.Random(), num_words)
    
        #check if the last tag is not a forbidden tag
        if generate_sent[-1][1] in forbidden_end_tags:
            continue  
        
        #get only words and connect them
        generate_sent = " ".join(words[0] for words, tag in generate_sent)
        generated_sentences.append(generate_sent)
        
    return generated_sentences
#the generated sentences are quite poor gramatically, 
#we generate 10 000 of them and then apply filtering techniques to find the best ones

basic_tri_sents = gen_basic_tri_sents(avg_sent_length, HMM_tri_tagger, 20)
print(basic_tri_sents)

['given so of cup where him would roof by a mouth jaime', 'left thought and at ache and lords a four blood', 'ill back to defend hostage he would give calumny', 'she sat bran crab grew a manses and been to be his defeat if third', 'a i you ships her thank', 't on dragon of your good loras was', 'the dragon if puddle said the sweet and white master will', 'you mount she wore to', 'the cruel with last dragonglass but king he to do that is', 'here to stark the clasping out your pale', "nor father's of a called piteously your men guarded to cape", "sometime this tyrion barristan that the wedding desmond quiet commander's", 'could give melted it of folded poorly ser scent shrill yet interested littler champions', 'no thin air when some behind hedge and rose even', 'i the bidding the peasant', 'that help the sealskins garbed a news rose again unspeakable realm rubbed only other', 'the tale and the river have last it so said most arrangements and the ale', 'and hard surcoat of the dany', 'one

In [32]:
import dill
with open("gen_basic_tri_sents.dill", 'wb') as file:
    dill.dump(gen_basic_tri_sents, file)
import pickle
with open("basic_tri_Sents.pkl", 'wb') as file:
    pickle.dump(basic_tri_sents, file)

In [52]:
print(HMM_tri_tagger._symbols[:10])

[('prologue', 'START', 'START'), ('we', 'prologue', 'START'), ('should', 'we', 'prologue'), ('start', 'should', 'we'), ('back', 'start', 'should'), ('gared', 'back', 'start'), ('urged', 'gared', 'back'), ('as', 'urged', 'gared'), ('the', 'as', 'urged'), ('woods', 'the', 'as')]
